# InternVL3-2B eval — Drive edition (apples-to-apples vs Claude)

Inference-only notebook for the trained InternVL3-2B LoRA. Same data + same
methodology as the Qwen eval (`eval_qwen_drive_v2.ipynb`) so results are
directly comparable in `make eval-replay`.

## One-time Drive setup

```
MyDrive/ttb_sft/
├── internvl3_2b/
│   └── adapter/                ← from sft_internvl3_2b.ipynb training run
└── eval/
    └── cola_sample.csv         ← already there from the Qwen eval setup
```

## How to run

1. Runtime → **A100 + High-RAM** (T4 works too, but A100 is faster)
2. Disconnect and delete runtime
3. Runtime → **Run all** (1st click — installs, kernel auto-crashes)
4. Runtime → **Run all** (2nd click — Drive permission dialog → accept; runs end-to-end)
5. `internvl3_outputs.jsonl` auto-downloads. Then locally:
   ```
   make eval-replay-internvl3
   ```

## Expected speed

InternVL3-2B at 2B params should be **~3-4× faster than Qwen 7B** on the
same A100. Target latency: ~1-2s/image, total ~3 min for 90 images.

## 1. Install dependencies

In [ ]:
# Same verified-working stack as the Qwen eval notebook + InternVL3 deps
# (timm + einops for the vision backbone).
import importlib.util, subprocess, sys, os


def _have(mod): return importlib.util.find_spec(mod) is not None
def _pinned(mod, want):
    try: return __import__(mod).__version__ == want
    except Exception: return False
def _numpy_ok():
    try:
        import numpy
        p = numpy.__version__.split(".")
        return int(p[0]) == 2 and int(p[1]) < 2
    except Exception: return False
def _torchao_ok():
    try:
        import torchao
        major, minor = [int(x) for x in torchao.__version__.split(".")[:2]]
        return (major, minor) >= (0, 16)
    except Exception: return False


_required = ["peft", "accelerate", "timm", "einops", "sentencepiece", "PIL"]
_ready = (_numpy_ok() and _pinned("transformers", "4.51.3")
          and _torchao_ok() and all(_have(m) for m in _required))

if _ready:
    import numpy, transformers
    print(f"✓ Set up — numpy {numpy.__version__}, transformers {transformers.__version__}")
else:
    print("⏳ Installing dependencies (~3 min). Kernel will auto-restart at end.")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "--upgrade", "pip"])
    subprocess.check_call([sys.executable, "-m", "pip", "install",
        "transformers==4.51.3", "accelerate>=0.30",
        "peft>=0.13", "torchao>=0.16",
        "protobuf>=5.27",
        "timm", "einops", "sentencepiece", "pillow", "tqdm"])
    subprocess.check_call([sys.executable, "-m", "pip", "install",
        "--force-reinstall", "--no-deps", "numpy>=2.0,<2.2"])
    print()
    print("=" * 70)
    print("✅ Install complete. Auto-restarting kernel.")
    print("   Wait for 'session crashed' banner, then Runtime → Run all AGAIN.")
    print("=" * 70)
    os.kill(os.getpid(), 9)

## 2. Mount Drive + verify the adapter and CSV are in place

In [ ]:
from google.colab import drive
from pathlib import Path

drive.mount("/content/drive")

DRIVE_ROOT  = Path("/content/drive/MyDrive/ttb_sft")
ADAPTER_DIR = DRIVE_ROOT / "internvl3_2b" / "adapter"
CSV_PATH    = DRIVE_ROOT / "eval" / "cola_sample.csv"
OUTPUT_DIR  = DRIVE_ROOT / "eval"

if not ADAPTER_DIR.exists() or not (ADAPTER_DIR / "adapter_config.json").exists():
    raise FileNotFoundError(
        f"\nMissing LoRA adapter at {ADAPTER_DIR}\n"
        f"Run sft_internvl3_2b.ipynb first to produce it."
    )
if not CSV_PATH.exists():
    raise FileNotFoundError(
        f"\nMissing CSV at {CSV_PATH}\n"
        f"Upload test/eval/data/cola_sample.csv to MyDrive/ttb_sft/eval/."
    )

print(f"✓ Adapter: {ADAPTER_DIR}")
print(f"✓ CSV:     {CSV_PATH} ({CSV_PATH.stat().st_size:,} bytes)")
print(f"✓ Output:  {OUTPUT_DIR}")

## 3. Load InternVL3-2B base + apply LoRA + merge

First load downloads `OpenGVLab/InternVL3-2B` (~4 GB BF16, ~1 min).

In [ ]:
import torch, time
from transformers import AutoModel, AutoTokenizer
from peft import PeftModel

BASE_MODEL = "OpenGVLab/InternVL3-2B"

print(f"Loading {BASE_MODEL} in BF16...")
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL, trust_remote_code=True, use_fast=False)
base = AutoModel.from_pretrained(
    BASE_MODEL,
    torch_dtype=torch.bfloat16,
    low_cpu_mem_usage=True,
    trust_remote_code=True,
).cuda()
base.eval()

print(f"Applying LoRA from {ADAPTER_DIR}...")
model = PeftModel.from_pretrained(base, str(ADAPTER_DIR))
model.eval()

print("Merging LoRA into base (safe in BF16)...")
model = model.merge_and_unload()
print(f"✓ Model ready. Free VRAM: {torch.cuda.mem_get_info()[0]/1e9:.1f} GB")

## 4. Load CSV + download test images from the COLA Cloud CDN

In [ ]:
import csv, urllib.request, concurrent.futures

rows = list(csv.DictReader(open(CSV_PATH)))
print(f"Loaded {len(rows)} rows")

IMG_DIR = Path("/content/internvl3_eval_images")
IMG_DIR.mkdir(exist_ok=True)
CDN = "https://dyuie4zgfxmt6.cloudfront.net/{}.webp"

def _dl(row):
    image_id = row["ttb_image_id"]
    dest = IMG_DIR / f"{image_id}.webp"
    if dest.exists() and dest.stat().st_size > 0:
        return row, True
    try:
        req = urllib.request.Request(CDN.format(image_id),
                                     headers={"User-Agent": "ttb-eval/0.1"})
        with urllib.request.urlopen(req, timeout=20) as r:
            dest.write_bytes(r.read())
        return row, True
    except Exception as e:
        print(f"  download fail {image_id}: {e}")
        return row, False

downloaded = []
with concurrent.futures.ThreadPoolExecutor(max_workers=16) as ex:
    for row, ok in ex.map(_dl, rows):
        if ok:
            row["_image_path"] = str(IMG_DIR / f"{row['ttb_image_id']}.webp")
            downloaded.append(row)
print(f"✓ Downloaded {len(downloaded)} / {len(rows)} images")

## 5. Run InternVL3 extraction on each image

In [ ]:
import json, re, statistics
from PIL import Image
import torchvision.transforms as T
from torchvision.transforms.functional import InterpolationMode

SYSTEM_PROMPT = (
    "You are a careful transcription assistant for U.S. TTB alcohol label review. "
    "Given ONE label panel image and the beverage type, READ what is printed and "
    "RETURN ONLY a JSON object matching the schema below. Do NOT decide compliance.\n\n"
    "If a field is not clearly visible, OMIT it from the fields object. Never guess; "
    "never substitute deposit codes (e.g. \"CA CRV\"), NOM IDs, or barcodes. Transcribe "
    "the Government Warning EXACTLY as printed (preserve case, punctuation, errors).\n\n"
    "Schema: {fields: {<field name>: {value, confidence}}, government_warning: "
    "{present, detected_text, casing_all_caps, heading_bold, body_bold, approx_font_mm, "
    "contrast_ok, separate_and_apart}, image_quality: {score, legible, note}}"
)

# InternVL3 image preprocessing — matches the training pipeline
IMAGENET_MEAN = (0.485, 0.456, 0.406)
IMAGENET_STD  = (0.229, 0.224, 0.225)

_img_transform = T.Compose([
    T.Lambda(lambda img: img.convert("RGB") if img.mode != "RGB" else img),
    T.Resize((448, 448), interpolation=InterpolationMode.BICUBIC),
    T.ToTensor(),
    T.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
])

generation_config = dict(max_new_tokens=512, do_sample=False)


@torch.no_grad()
def _internvl3_extract(image_path, beverage_type):
    img = Image.open(image_path).convert("RGB")
    pixel_values = _img_transform(img).unsqueeze(0).to(torch.bfloat16).cuda()
    question = f"<image>\nBeverage type: {beverage_type}. {SYSTEM_PROMPT}"
    try:
        response = model.chat(tokenizer, pixel_values, question, generation_config)
    except AttributeError:
        # In case the merge_and_unload changed the API surface
        response = model.base_model.chat(tokenizer, pixel_values, question, generation_config)
    s, e = response.find("{"), response.rfind("}")
    if s == -1 or e == -1:
        return None, response
    try:
        return json.loads(response[s:e+1]), response
    except json.JSONDecodeError:
        cleaned = re.sub(r",(\s*[}\]])", r"\1", response[s:e+1])
        try: return json.loads(cleaned), response
        except Exception: return None, response


print("Sanity check on first image (warms CUDA kernels)...")
t0 = time.time()
pred0, _ = _internvl3_extract(downloaded[0]["_image_path"],
                               (downloaded[0].get("beverage_type") or "wine").lower())
print(f"  ✓ First image: {time.time()-t0:.2f}s")

outputs_jsonl = []
n_parsed = n_unparseable = 0
latencies = []

print(f"\nExtracting on {len(downloaded)} images...")
for i, row in enumerate(downloaded, 1):
    bev = (row.get("beverage_type") or "wine").lower()
    t0 = time.time()
    pred, raw_text = _internvl3_extract(row["_image_path"], bev)
    dt = time.time() - t0
    latencies.append(dt)
    outputs_jsonl.append({
        "image_filename": row.get("image_filename") or (row["ttb_image_id"] + ".webp"),
        "ttb_image_id":   row["ttb_image_id"],
        "beverage_type":  bev,
        "raw_output":     raw_text,
        "parsed_output":  pred,
        "latency_sec":    round(dt, 3),
    })
    if pred is None: n_unparseable += 1
    else:            n_parsed += 1
    if i % 10 == 0:
        print(f"  [{i}/{len(downloaded)}] parsed={n_parsed} unparseable={n_unparseable} "
              f"latency_mean={statistics.mean(latencies):.2f}s")

print()
print("=" * 70)
print(f"Done. Parsed: {n_parsed} / Unparseable: {n_unparseable}")
print(f"Latency mean / p95: {statistics.mean(latencies):.2f}s / "
      f"{sorted(latencies)[int(len(latencies)*0.95)]:.2f}s")
print("=" * 70)

## 6. Save outputs to Drive AND auto-download to your Mac

In [ ]:
jsonl_path = OUTPUT_DIR / "internvl3_outputs.jsonl"
with open(jsonl_path, "w") as f:
    for rec in outputs_jsonl:
        f.write(json.dumps(rec) + "\n")
print(f"✓ Saved {jsonl_path} ({len(outputs_jsonl)} rows)")

from google.colab import files
files.download(str(jsonl_path))

print()
print("Next on your Mac:")
print("  make eval-replay-internvl3")
print("    → writes test/eval/internvl3_report.json + side-by-side vs Claude.")